## Setup

### Importing libraries and datasets

In [40]:
# %% [Cell 1] — Imports & Load
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("combined_dataset.csv")
df['label'] = (df['target'] == 'spam').astype(int)

print(df.shape)
print(df['target'].value_counts())

(10961, 3)
target
ham     8555
spam    2406
Name: count, dtype: int64


### Training data split

Split: 70% 15% 15%

In [41]:

X_temp, X_test, y_temp, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.15,
    random_state=42,
    stratify=df['label']
)


X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train)} | Dev: {len(X_dev)} | Test: {len(X_test)}")
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Dev class balance:\n", y_dev.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train: 7676 | Dev: 1640 | Test: 1645
Train class balance:
 label
0    0.780485
1    0.219515
Name: proportion, dtype: float64
Dev class balance:
 label
0    0.780488
1    0.219512
Name: proportion, dtype: float64
Test class balance:
 label
0    0.780547
1    0.219453
Name: proportion, dtype: float64


### Vectorization


In [42]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,3), min_df=3, max_df=0.9)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_dev_tfidf = vectorizer.transform(X_dev)
X_test_tfidf = vectorizer.transform(X_test)

## Training


### Naive Bayes

In [43]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

MultinomialNB()

In [44]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=3, penalty="l2")
lr.fit(X_train_tfidf, y_train)

/Users/arqies/Documents/Githubs/Spam-Classifier/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/arqies/Documents/Githubs/Spam-Classifier/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/arqies/Documents/Githubs/Spam-Classifier/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


LogisticRegression(C=3, class_weight='balanced', max_iter=1000)

### Deciding best model

In [45]:
nb_train_preds = nb.predict(X_train_tfidf)
nb_dev_preds = nb.predict(X_dev_tfidf)

lr_train_preds = lr.predict(X_train_tfidf)
lr_dev_preds = lr.predict(X_dev_tfidf)

print("=== Naive Bayes (Train) ===")
print("Accuracy:", accuracy_score(y_train, nb_train_preds))
print(classification_report(y_train, nb_train_preds, target_names=['ham', 'spam']))

print("=== Naive Bayes (Dev) ===")
print("Accuracy:", accuracy_score(y_dev, nb_dev_preds))
print(classification_report(y_dev, nb_dev_preds, target_names=['ham', 'spam']))

print("=== Logistic Regression (Train) ===")
print("Accuracy:", accuracy_score(y_train, lr_train_preds))
print(classification_report(y_train, lr_train_preds, target_names=['ham', 'spam']))

print("=== Logistic Regression (Dev) ===")
print("Accuracy:", accuracy_score(y_dev, lr_dev_preds))
print(classification_report(y_dev, lr_dev_preds, target_names=['ham', 'spam']))

=== Naive Bayes (Train) ===
Accuracy: 0.9572694111516414
              precision    recall  f1-score   support

         ham       0.97      0.98      0.97      5991
        spam       0.91      0.89      0.90      1685

    accuracy                           0.96      7676
   macro avg       0.94      0.93      0.94      7676
weighted avg       0.96      0.96      0.96      7676

=== Naive Bayes (Dev) ===
Accuracy: 0.9414634146341463
              precision    recall  f1-score   support

         ham       0.95      0.97      0.96      1280
        spam       0.90      0.82      0.86       360

    accuracy                           0.94      1640
   macro avg       0.93      0.90      0.91      1640
weighted avg       0.94      0.94      0.94      1640

=== Logistic Regression (Train) ===
Accuracy: 0.977853048462741
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99      5991
        spam       0.93      0.98      0.95      1685

    ac

## Test (Chosen Logistic regression for better recall and precision)

In [47]:
# %% [Cell 7] — Final Test Evaluation: Logistic Regression (run ONLY once)
lr_test_preds = lr.predict(X_test_tfidf)

print("=== Logistic Regression (Test) ===")
print("Accuracy:", accuracy_score(y_test, lr_test_preds))
print(classification_report(y_test, lr_test_preds, target_names=['ham', 'spam']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_test_preds))

=== Logistic Regression (Test) ===
Accuracy: 0.9404255319148936
              precision    recall  f1-score   support

         ham       0.97      0.95      0.96      1284
        spam       0.84      0.89      0.87       361

    accuracy                           0.94      1645
   macro avg       0.91      0.92      0.91      1645
weighted avg       0.94      0.94      0.94      1645

Confusion Matrix:
[[1224   60]
 [  38  323]]
